In [2]:
import pandas as pd

df = pd.read_csv("stock_prices/all_stocks_with_returns.csv")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Stock_symbol", "Date"]
).reset_index(drop=True)

print("Total rows:", len(df))
print("Companies:", df["Stock_symbol"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())

display(df.head())

Total rows: 14084
Companies: 14
Date range: 2020-01-02 00:00:00 to 2023-12-29 00:00:00


,Date,Stock_symbol,Open,High,Low,Close,Adj Close,Volume,Next_Adj_Close,Future_Return
0,2020-01-02,AAPL,74.059998,75.150002,73.797501,75.087502,72.271545,135480400,71.568901,-0.009722
1,2020-01-03,AAPL,74.287498,75.144997,74.125000,74.357498,71.568901,146322800,72.139198,0.007969
2,2020-01-06,AAPL,73.447502,74.989998,73.187500,74.949997,72.139198,118387200,71.799927,-0.004703
3,2020-01-07,AAPL,74.959999,75.224998,74.370003,74.597504,71.799927,108872000,72.954895,0.016086
4,2020-01-08,AAPL,74.290001,76.110001,74.290001,75.797501,72.954895,132079200,74.504562,0.021241


In [4]:
# Split the dataset by time

train = df[df["Date"] < "2023-01-01"].copy()

val = df[
    (df["Date"] >= "2023-01-01") &
    (df["Date"] < "2023-07-01")
].copy()

test = df[df["Date"] >= "2023-07-01"].copy()

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nTrain dates:", train["Date"].min(), "to", train["Date"].max())
print("Validation dates:", val["Date"].min(), "to", val["Date"].max())
print("Test dates:", test["Date"].min(), "to", test["Date"].max())

Train: 10584
Validation: 1736
Test: 1764

Train dates: 2020-01-02 00:00:00 to 2022-12-30 00:00:00
Validation dates: 2023-01-03 00:00:00 to 2023-06-30 00:00:00
Test dates: 2023-07-03 00:00:00 to 2023-12-29 00:00:00


In [6]:
# Step 3: Create price-volume features

df = df.sort_values(
    ["Stock_symbol", "Date"]
).copy()

group = df.groupby("Stock_symbol")

# Daily return
df["Return_1d"] = group["Adj Close"].pct_change(
    fill_method=None
)

# 5-day return
df["Return_5d"] = (
    df["Adj Close"] /
    group["Adj Close"].shift(5) - 1
)

# Average trading volume over the last 5 days
df["Volume_MA5"] = (
    group["Volume"]
    .transform(lambda x: x.rolling(5).mean())
)

# Relative trading volume
df["Volume_Ratio"] = df["Volume"] / df["Volume_MA5"]

# 5-day volatility
df["Volatility_5d"] = (
    df.groupby("Stock_symbol")["Return_1d"]
    .transform(lambda x: x.rolling(5).std())
)

features = [
    "Return_1d",
    "Return_5d",
    "Volume_Ratio",
    "Volatility_5d"
]

display(
    df[["Date", "Stock_symbol"] + features + ["Future_Return"]]
    .head(10)
)

print("\nMissing values in features:")
print(df[features].isna().sum())

,Date,Stock_symbol,Return_1d,Return_5d,Volume_Ratio,Volatility_5d,Future_Return
0,2020-01-02,AAPL,NaN,NaN,NaN,NaN,-0.009722
1,2020-01-03,AAPL,-0.009722,NaN,NaN,NaN,0.007969
2,2020-01-06,AAPL,0.007969,NaN,NaN,NaN,-0.004703
3,2020-01-07,AAPL,-0.004703,NaN,NaN,NaN,0.016086
4,2020-01-08,AAPL,0.016086,NaN,1.030031,NaN,0.021241
5,2020-01-09,AAPL,0.021241,0.030898,1.258627,0.013224,0.002260
6,2020-01-10,AAPL,0.002260,0.043372,1.049445,0.010409,0.021364
7,2020-01-13,AAPL,0.021364,0.057238,0.902595,0.011841,-0.013503
8,2020-01-14,AAPL,-0.013503,0.047890,1.114899,0.015029,-0.004286
9,2020-01-15,AAPL,-0.004286,0.026881,0.851228,0.015547,0.012526



Missing values in features:
Return_1d        14
Return_5d        70
Volume_Ratio     56
Volatility_5d    70
dtype: int64


In [8]:
# Step 4: Prepare model-ready datasets

model_df = df.dropna(
    subset=features + ["Future_Return"]
).copy()

train = model_df[
    model_df["Date"] < "2023-01-01"
].copy()

val = model_df[
    (model_df["Date"] >= "2023-01-01") &
    (model_df["Date"] < "2023-07-01")
].copy()

test = model_df[
    model_df["Date"] >= "2023-07-01"
].copy()

print("Total usable rows:", len(model_df))
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nMissing values:")
print(model_df[features + ["Future_Return"]].isna().sum())

Total usable rows: 14000
Train: 10514
Validation: 1736
Test: 1750

Missing values:
Return_1d        0
Return_5d        0
Volume_Ratio     0
Volatility_5d    0
Future_Return    0
dtype: int64


In [10]:
# Step 5: Remove rows whose target crosses split boundaries

train = train[
    train["Date"] < "2022-12-30"
].copy()

val = val[
    val["Date"] < "2023-06-30"
].copy()

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nTrain last date:", train["Date"].max())
print("Validation last date:", val["Date"].max())
print("Test last date:", test["Date"].max())

Train: 10500
Validation: 1722
Test: 1750

Train last date: 2022-12-29 00:00:00
Validation last date: 2023-06-29 00:00:00
Test last date: 2023-12-28 00:00:00


In [16]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Model inputs
X_train = train[features]
y_train = train["Future_Return"]

X_val = val[features]
y_val = val["Future_Return"]

# StandardScaler + Ridge Regression
model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

# Train only on historical training data
model.fit(X_train, y_train)

# Predict validation returns
val_pred = model.predict(X_val)

print("Model trained successfully!")
print("Training samples:", len(X_train))
print("Validation predictions:", len(val_pred))

display(
    pd.DataFrame({
        "Stock_symbol": val["Stock_symbol"].values,
        "Date": val["Date"].values,
        "Actual_Return": y_val.values,
        "Predicted_Return": val_pred
    }).head(10)
)

Model trained successfully!
Training samples: 10500
Validation predictions: 1722


,Stock_symbol,Date,Actual_Return,Predicted_Return
0,AAPL,2023-01-03,0.010314,0.003003
1,AAPL,2023-01-04,-0.010605,-0.000245
2,AAPL,2023-01-05,0.036794,0.001602
3,AAPL,2023-01-06,0.004089,-0.001916
4,AAPL,2023-01-09,0.004456,0.000623
5,AAPL,2023-01-10,0.021113,0.001140
6,AAPL,2023-01-11,-0.000600,-0.000037
7,AAPL,2023-01-12,0.010119,0.001719
8,AAPL,2023-01-13,0.008757,0.000702
9,AAPL,2023-01-17,-0.005370,0.000825


In [14]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 53.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 59.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [18]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Evaluate validation predictions
mae = mean_absolute_error(y_val, val_pred)

rmse = np.sqrt(
    mean_squared_error(y_val, val_pred)
)

direction_acc = np.mean(
    np.sign(y_val.to_numpy()) == np.sign(val_pred)
)

print("Validation Results:")
print(f"MAE: {mae:.6f} ({mae * 100:.3f}%)")
print(f"RMSE: {rmse:.6f} ({rmse * 100:.3f}%)")
print(f"Directional Accuracy: {direction_acc:.2%}")

Validation Results:
MAE: 0.015337 (1.534%)
RMSE: 0.022173 (2.217%)
Directional Accuracy: 51.74%


In [20]:
from pathlib import Path

output_dir = Path("baseline_results")
output_dir.mkdir(exist_ok=True)

baseline_results = val[
    ["Date", "Stock_symbol", "Future_Return"]
].copy()

baseline_results["Predicted_Return"] = val_pred

baseline_results = baseline_results.rename(
    columns={"Future_Return": "Actual_Return"}
)

baseline_results.to_csv(
    output_dir / "price_volume_predictions.csv",
    index=False
)

print("Saved successfully!")
print("Rows:", len(baseline_results))

Saved successfully!
Rows: 1722


In [22]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Notebook results:")

for file in Path("baseline_results").glob("*.csv"):
    print(file, file.stat().st_size, "bytes")

Current folder: /Users/yuky/Desktop/LLM/Project/DataCollect
Notebook results:
baseline_results/price_volume_predictions.csv 97340 bytes
